# Cubic triangles and edge orientation

This notebook accompanies Exercise 27.

Determine the three basis-function normalizations from the sheet and complete `EXERCISE_P3_SHAPES`, `EXERCISE_P3_DSHAPES`, `EXERCISE_P3_NUMBERING` and `EXERCISE_EDGE_ORIENTATION` in `implementation_ueb4/`. Keep a fixed local nodal basis and handle edge orientation in `GetDofNrs`. The supplied $P_1$ classes provide the model, and the cubic boundary segment, NGSolve coupling bookkeeping, compilation, bindings and meshes are supplied.

The nodality, dimension, cubic-reproduction and convergence cells are ready to run after completing the C++ parts. Fill only the two short coefficient tasks in the trace experiment and interpret the results.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from ngsolve import *
from ngsolve.fem import CompilePythonModule
from ngsolve.webgui import Draw
from netgen.meshing import Mesh as NetgenMesh, MeshPoint, Pnt, Element2D, Element1D

lecture_dir = Path("../../notebooks/ch06_implementation").resolve()
sys.path.insert(0, str(lecture_dir))
from teaching_utils import square_mesh, max_diameter, errors
m = CompilePythonModule(Path("implementation_ueb4/mymodule.cpp").resolve(),
                        init_function_name="teaching_fem")

## 27. A nodal $P_3$ element and edge orientation

A cubic triangle has three vertex nodes, two trisection nodes on every edge and one barycentre node. Determine the constants in the three basis-function families given on the sheet, then implement values and reference derivatives. Analytical derivatives or AutoDiff may be used. The local edge order is $(2,0),(1,2),(0,1)$; on an edge $(a,b)$ its nodes are $(2v_a+v_b)/3$ and $(v_a+2v_b)/3$.

Run the following nodality test after completing part (a).

In [ ]:
nodes_p3 = m.ReferenceNodes("triangle", 3)
values_p3 = []
for point in nodes_p3:
    shape, _ = m.ReferenceShapes("triangle", 3, point)
    values_p3.append(shape)
values_p3 = np.array(values_p3)
print(f"triangle nodality error: {np.linalg.norm(values_p3 - np.eye(10)):.3e}")

### Global numbering and dimension

Complete the total dimension and the cell-interior query in `myP3TriangleFESpace.hpp`, then implement the edge-pair permutation in `GetDofNrs`. Coupling bookkeeping and the edge query are supplied. Global edges point from the smaller to the larger global vertex number. The supplied cubic segment uses endpoint nodes $1,0$, followed by $2/3,1/3$.

After completing part (b), run the dimension test before boundary constraints

$$
\begin{align*}
N=N_V+2N_E+N_T.
\end{align*}
$$



In [ ]:
mesh_p3_count = square_mesh(2)
fes_p3_count = m.P3TriangleFESpace(mesh_p3_count)
print("P3 coefficients and Nv+2Ne+NT:", fes_p3_count.ndof,
      mesh_p3_count.nv + 2 * mesh_p3_count.nedge + mesh_p3_count.ne)

### Compare the two traces directly

The supplied patch has two counterclockwise triangles with vertex lists $(0,1,2)$ and $(3,2,1)$. Their common local edge points in opposite directions. It has local edge number 1 (zero-based), with local endpoints $(1,2)$, on both triangles. Set the two shared-edge nodal values to $+1,-1$ in global edge order and all other coefficients to zero. This isolates the odd cubic edge bubble, so the orientation error is easy to see. Compare the traces at the **same physical points**.

Pull each point back using the Jacobian with columns $v_0-v_2,v_1-v_2$. Obtain the correct local coefficients from `GetDofNrs`. To simulate omitting the orientation correction, undo the shared-edge pair permutation on any triangle whose local shared edge runs against the global direction.

Why does a nodal $P_2$ element with one midpoint per edge hide this problem? For hierarchical edge modes, reversal leaves the even quadratic mode unchanged and changes the sign of the odd cubic mode. Relate this to the symmetric and antisymmetric combinations $N_a+N_b$ and $N_a-N_b$ of the two nodal edge shapes.

In [ ]:
patch = NetgenMesh(dim=2)
patch_domain = patch.AddRegion("domain", dim=2)
patch_boundary = patch.AddRegion("boundary", dim=1)
patch_points = [patch.Add(MeshPoint(Pnt(a, b, 0)))
                for a, b in [(0, 0), (1, 0), (0, 1), (1, 1)]]
for indices in [(0, 1, 2), (3, 2, 1)]:
    patch.Add(Element2D(patch_domain, [patch_points[i] for i in indices]))
for a, b in [(0, 1), (1, 3), (3, 2), (2, 0)]:
    patch.Add(Element1D([patch_points[a], patch_points[b]], index=patch_boundary))
trace_mesh = Mesh(patch)
trace_fes = m.P3TriangleFESpace(trace_mesh)
edge_pairs = [(2, 0), (1, 2), (0, 1)]

In [ ]:
trace_cells = list(trace_mesh.Elements())
cell_vertices = [[vertex.nr for vertex in element.vertices] for element in trace_cells]
coordinates = np.array([vertex.point[:2] for vertex in trace_mesh.vertices])
shared_vertices = [1, 2]
shared_local_edge = 1
for edge in trace_mesh.edges:
    edge_vertices = [vertex.nr for vertex in edge.vertices]
    if sorted(edge_vertices) == shared_vertices:
        shared_edge = edge.nr
        break
shared_edge_dofs = list(trace_fes.GetDofNrs(NodeId(EDGE, shared_edge)))
global_coefficients = np.zeros(trace_fes.ndof)
global_coefficients[shared_edge_dofs] = [1.0, -1.0]
edge_start = coordinates[shared_vertices[0]]
edge_end = coordinates[shared_vertices[1]]
sample_t = np.linspace(0, 1, 41)

In [ ]:
trace_values, broken_trace_values, local_shared_directions = [], [], []
broken_signs = []
for element, numbers in zip(trace_cells, cell_vertices):
    vertices = coordinates[numbers]
    J = np.column_stack((vertices[0] - vertices[2], vertices[1] - vertices[2]))
    a, b = edge_pairs[shared_local_edge]
    local_shared_directions.append((numbers[a], numbers[b]))
    dofs = list(trace_fes.GetDofNrs(element))
    # TODO: read the local coefficients using the oriented global dof list.
    local_coefficients = None
    broken_coefficients = local_coefficients.copy()
    if numbers[a] > numbers[b]:
        indices = [3 + 2 * shared_local_edge, 4 + 2 * shared_local_edge]
        # TODO: undo the pair permutation on this reversed local edge.
        pass
    # For the supplied WebGUI views: swapping this pure odd bubble negates it.
    edge_index = 3 + 2 * shared_local_edge
    broken_signs.append(broken_coefficients[edge_index] / local_coefficients[edge_index])
    values, broken_values = [], []
    for t in sample_t:
        point = (1 - t) * edge_start + t * edge_end
        reference_point = np.linalg.solve(J, point - vertices[2])
        shape, _ = m.ReferenceShapes("triangle", 3, reference_point.tolist())
        shape = np.array(shape)
        values.append(shape @ local_coefficients)
        broken_values.append(shape @ broken_coefficients)
    trace_values.append(np.array(values))
    broken_trace_values.append(np.array(broken_values))

In [ ]:
trace_jump = np.max(abs(trace_values[0] - trace_values[1]))
broken_trace_jump = np.max(abs(broken_trace_values[0] - broken_trace_values[1]))
print("local shared-edge directions:", local_shared_directions)
print(f"correct trace jump: {trace_jump:.3e}")
print(f"jump without the permutation: {broken_trace_jump:.3e}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for side in range(2):
    axes[0].plot(sample_t, trace_values[side], "-" if side == 0 else "--", label=f"triangle {side}")
    axes[1].plot(sample_t, broken_trace_values[side], "-" if side == 0 else "--", label=f"triangle {side}")
for ax, title in zip(axes, ["With orientation correction", "Without orientation correction"]):
    ax.set_title(title)
    ax.set_xlabel("global edge parameter t")
    ax.set_ylabel("trace value")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

### View the cubic edge bubble

Both views use the field value as height and color, with the same scale. The correctly oriented surfaces meet along the shared edge. Without the permutation, one triangle has the opposite trace.

For this pure odd bubble, swapping the two nodal values negates the entire element polynomial. The supplied visualization uses a piecewise constant $L^2$ sign factor to retain that jump.

In [ ]:
trace_field = GridFunction(trace_fes)
trace_field.vec.FV().NumPy()[:] = global_coefficients
sign_fes = L2(trace_mesh, order=0)
broken_sign = GridFunction(sign_fes)
for element, sign in zip(trace_cells, broken_signs):
    sign_dof = sign_fes.GetDofNrs(element)[0]
    broken_sign.vec[sign_dof] = sign
broken_field = broken_sign * trace_field

print("With orientation correction")
correct_scene = Draw(trace_field, trace_mesh, deformation=True, scale=0.3,
                     min=-1.4, max=1.4, order=6, euler_angles=[-60, 0, -35],
                     settings={"subdivision": 8})
print("Without orientation correction")
broken_scene = Draw(broken_field, trace_mesh, deformation=True, scale=0.3,
                    min=-1.4, max=1.4, order=6, euler_angles=[-60, 0, -35],
                    settings={"subdivision": 8})

### Cubic reproduction and Dirichlet lifting

The supplied driver and call below use

$$
\begin{align*}
u=x^3+2xy^2,\qquad -\Delta u+u=-10x+u,
\end{align*}
$$

on the unit square with its exact Dirichlet trace. The driver assembles $\mathcal A(u_h,v_h)=\int_\Omega(\nabla u_h\cdot\nabla v_h+cu_hv_h)$ and $F(v_h)=\int_\Omega fv_h$. It sets the boundary trace first and then solves the residual equation on `FreeDofs()`. Here $c=1$.

Run the supplied cells and explain the $L^2$ and full $H^1$ errors.

In [ ]:
def solve_case(n, exact, source, reaction):
    mesh = square_mesh(n)
    fes = m.P3TriangleFESpace(mesh, dirichlet=".*")
    u, v = fes.TnT()
    a = BilinearForm((grad(u) * grad(v) + reaction * u * v) * dx(bonus_intorder=6)).Assemble()
    f = LinearForm(source * v * dx(bonus_intorder=6)).Assemble()
    uh = GridFunction(fes)
    uh.Set(exact, definedon=mesh.Boundaries(".*"))
    rhs = f.vec.CreateVector()
    rhs.data = f.vec - a.mat * uh.vec
    uh.vec.data += a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * rhs
    error = errors(mesh, uh, exact, quadrature_order=16)
    result = {"n": n, "h": max_diameter(mesh), "ndof": fes.ndof,
              "L2": error["L2"], "H1": error["H1"]}
    return mesh, fes, uh, result

In [ ]:
cubic = x**3 + 2 * x * y**2
mesh_p3, fes_p3, uh_p3, result_p3 = solve_case(
    3, exact=cubic, source=-10 * x + cubic, reaction=1)
print(f'cubic reproduction: L2={result_p3["L2"]:.3e}, H1={result_p3["H1"]:.3e}')

### Smooth-solution convergence

The supplied cell solves the homogeneous Dirichlet problem with $u=\sin(\pi x)\sin(\pi y)$ and $f=2\pi^2u$ on meshes with $n=4,8,16$. It records the actual maximum element diameter $h$, computes both errors and the consecutive rates

$$
\begin{align*}
\operatorname{EOC}=\frac{\log(E_{h_1}/E_{h_2})}{\log(h_1/h_2)},
\end{align*}
$$

and plots both error norms against $h$ on logarithmic axes.

In [ ]:
sine = sin(pi * x) * sin(pi * y)
convergence = []
for n in [4, 8, 16]:
    _, _, _, row = solve_case(n, exact=sine, source=2 * pi**2 * sine, reaction=0)
    convergence.append(row)
print(" n          h            L2            H1       EOC L2   EOC H1")
for i, row in enumerate(convergence):
    eoc_l2 = np.nan
    eoc_h1 = np.nan
    if i > 0:
        previous = convergence[i - 1]
        log_h_ratio = np.log(previous["h"] / row["h"])
        eoc_l2 = np.log(previous["L2"] / row["L2"]) / log_h_ratio
        eoc_h1 = np.log(previous["H1"] / row["H1"]) / log_h_ratio
    print(f'{row["n"]:2d}  {row["h"]:.4e}  {row["L2"]:.4e}  {row["H1"]:.4e}'
          f'    {eoc_l2:5.3f}    {eoc_h1:5.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, norm in zip(axes, ["L2", "H1"]):
    ax.loglog([row["h"] for row in convergence],
              [row[norm] for row in convergence], "o-")
    ax.set_xlabel("maximum element diameter h")
    ax.set_ylabel(norm + " error")
    ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()